# Lesson 17 Lab — OpenVINO, NNCF, and Intel Runtime Sparsity

**Puzzle:** Why can a generic sparse checkpoint miss the optimized CPU path?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

CPU deployment benefit depends on a pattern, graph transformation, precision, and operator implementation supported by the target OpenVINO/oneDNN stack. NNCF or Intel Neural Compressor configuration is part of the executable artifact; a PyTorch zero rate alone is not.


## 0. Predict before running

1. Predict which package probes succeed in the recorded GPU environment.
2. Explain why physically narrower shapes remain useful without a sparse CPU kernel.
3. List the CPU-specific fields needed for a fair benchmark.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

The notebook probes OpenVINO, NNCF, and Neural Compressor packages, creates unstructured and filter-pruned controls on CUDA, and records a deployment gate matrix without asserting CPU speed from GPU evidence.

- Framework zeros are not an OpenVINO execution plan.
- Filter removal and unstructured encoding expose different CPU opportunities.
- GPU control results cannot substitute for CPU runtime measurements.


## 2. Derive the mechanism

Unstructured zeros preserve dense tensor dimensions unless a sparse encoding and sparse operator are selected. NNCF filter pruning can propagate structural changes and export a smaller graph, while post-training sparsity tools may target runtime-specific patterns. CPU SIMD utilization, threading, cache behavior, and quantization interact with width. Therefore the correct handoff includes model format, pattern, runtime version, thread settings, and operator log.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 17
LESSON_TITLE = 'OpenVINO, NNCF, and Intel Runtime Sparsity'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260825
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | same-shape unstructured zero mask represented as a dense PyTorch tensor |
| Candidate | physically narrower dense control and optional OpenVINO/NNCF native path |
| Held constant | source tensor, zero budget, input, environment, package names, and decision gates |
| Measurements | package availability, logical sparsity, physical width, output drift, and native-run status |
| Evidence | `compatibility-probe` |

**Experiment:** Probe Intel compression/runtime packages and contrast value sparsity with physical width under a bounded evidence label.


## 5. Read the experiment code

The experiment keeps its CUDA numerical control separate from the package matrix. Conditional imports record exact availability; the conclusion remains `not_run` for OpenVINO performance unless a native model conversion and CPU workload execute. This prevents a generic pruning result from being laundered into an Intel deployment claim.

Do not execute until the code implements the frozen table above.


In [2]:
availability={name:(importlib.util.find_spec(name) is not None) for name in ("openvino","nncf","neural_compressor")}
w=torch.randn(1024,1024,device=DEVICE,dtype=torch.bfloat16); x=torch.randn(16,1024,device=DEVICE,dtype=torch.bfloat16)
masked=w*magnitude_mask(w,0.75); narrow=w[:256]
with torch.inference_mode(): full_y=F.linear(x,w); narrow_y=F.linear(x,narrow)
metrics={"openvino_available":availability["openvino"],"nncf_available":availability["nncf"],"neural_compressor_available":availability["neural_compressor"],"logical_sparsity":zero_fraction(masked),"original_width":1024,"narrow_width":256,"width_reduction":0.75,"full_output_shape":list(full_y.shape),"narrow_output_shape":list(narrow_y.shape),"native_cpu_run":False,"native_cpu_latency_ms":None}
analysis=(f"The dense-value control reached {metrics['logical_sparsity']:.1%} logical sparsity without changing its 1024-wide "
          f"output; the physical control changed width to 256. OpenVINO/NNCF/Neural Compressor availability was "
          f"{availability['openvino']}/{availability['nncf']}/{availability['neural_compressor']}. No CPU latency is reported because no native CPU path executed.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| OpenVINO available | no |
| NNCF available | no |
| Neural Compressor available | no |
| Logical sparsity | 75.00% |
| Physical width reduction | 75.00% |
| Native CPU run | no |


## 7. Interpret rather than merely print

The dense-value control reached 75.0% logical sparsity without changing its 1024-wide output; the physical control changed width to 256. OpenVINO/NNCF/Neural Compressor availability was False/False/False. No CPU latency is reported because no native CPU path executed.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The notebook records real package/API availability and preserves the native success or failure state. Missing backend execution remains unmeasured.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 17,
    "title": 'OpenVINO, NNCF, and Intel Runtime Sparsity',
    "environment": ENV,
    "evidence_label": 'compatibility-probe',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Intel sparsity is a runtime-specific graph and kernel decision; CUDA zeros provide only a numerical control.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 17,
  "title": "OpenVINO, NNCF, and Intel Runtime Sparsity",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260825
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "openvino_available": false,
    "nncf_available": false,
    "neural_compressor_available": false,
    "logical_sparsity": 0.75,
    "original_width": 1024,
    "narrow_width": 256,
    "width_reduction": 0.75,
    "full_output_shape": [
      16,
      1024
    ],
    "narrow_output_shape": [
      16,
      256
    ],
    "native_cpu_run": false,
    "native_cpu_latency_ms": null
  },
  "analysis": "The dense-value control reached 75.0% logical sparsity without changing its 1024-wide output; the physical control changed width to 256. OpenVINO/NNCF/Neural Compressor availability was False/False/False. No CPU latency is reported because no native CPU path execu

## 9. Make the bounded decision

> Intel sparsity is a runtime-specific graph and kernel decision; CUDA zeros provide only a numerical control.

**Acceptance/rollback:** Accept an Intel deployment only after conversion, graph inspection, CPU thread pinning, quality parity, and repeated target-CPU latency/throughput evidence.

**Failure analysis:** Package presence is weaker than operator support, and a laptop CPU result may not transfer to the production SKU. A narrow channel count can hurt vector alignment, while unstructured compression can reduce disk size without runtime benefit.


## 10. Extend the evidence

Create a pinned OpenVINO/NNCF environment, export both candidates, inspect IR dimensions and operators, then benchmark several thread and batch settings on the actual CPU target.

The full evidence boundary and references are in [`README.md`](README.md).
